# Tutorial: Plan 002 — Catalog Refresh and Reconciliation

**Audience:** developers using Archiver's Python API or `archiver catalog` command.

**Prerequisites:** launch Jupyter from this repository (`uv run jupyter lab notebooks`) and know basic Python/path handling.

By the end, you will be able to:

1. reconcile filesystem observations without changing a catalog;
2. inspect `new`, `unchanged`, `modified`, and `missing` changes;
3. apply a change set atomically and understand its stale-baseline guard; and
4. use the CLI to see the same refresh summary.

Every example uses a temporary directory and removes it at the end.


## Why refresh has three stages

Plan 002 keeps **observe/reconcile -> describe -> apply** separate.

- `reconcile_directory()` reads the filesystem and returns a `RefreshChangeSet`; it writes nothing to SQLite.
- The change set is a useful dry-run/comparison value, but it is not a filesystem lock or snapshot.
- `apply_refresh()` first checks that the catalog's current scan is still the baseline used for reconciliation, then writes one complete new snapshot atomically.

The baseline check detects **catalog staleness**—another successful refresh—not filesystem changes after reconciliation.


In [ ]:
from __future__ import annotations

import os
import shutil
import subprocess
import sys
import tempfile
from pathlib import Path
import archiver.pandas as arc_pandas
from archiver import Catalog, StaleRefreshError

workspace = Path(tempfile.mkdtemp(prefix="archiver-plan-002-"))
api_root = workspace / "api-source"
api_root.mkdir()
database_path = workspace / "catalog.sqlite"

(api_root / "stable.txt").write_bytes(b"stable bytes")
(api_root / "changed.txt").write_bytes(b"first version")
(api_root / "removed.txt").write_bytes(b"remove me")

catalog = Catalog.create(database_path)
print(f"Temporary workspace: {workspace}")
print("Initial files:", [path.name for path in sorted(api_root.iterdir())])


## 1. Reconcile an initial directory without writing

The first reconciliation sees every path as `new`. Notice that the current catalog remains empty until `apply_refresh()` is called.


In [ ]:
initial = catalog.reconcile_directory(api_root)

print("Baseline scan:", initial.baseline_scan_id)
print("Summary:", initial.summary)
print([(change.relative_path.as_posix(), change.kind, change.hash_reused) for change in initial.changes])
print("Current catalog before apply:", catalog.current_files(api_root))


## 2. Apply the change set atomically

Applying persists the complete current snapshot. The original files are only read; Archiver never renames, deletes, or rewrites them during a refresh.


In [ ]:
summary = catalog.apply_refresh(initial)
print("Applied scan summary:", summary)
print("Current paths:", [item.relative_path.as_posix() for item in catalog.current_files(api_root)])


## 3. Inspect all four change kinds

Now make one metadata-only change, one content change, one deletion, and one addition.

A matching path/size/`mtime_ns` reuses the prior content identity without hashing. If metadata changes, Archiver hashes the file again. When that hash is still the same, the result remains `unchanged`, with updated metadata.

Note: reconcile_directory does not apply any changes yet. Only after a apply_refresh will you see the changes documented in the catalog.

In [ ]:
# change some files
stable = api_root / "stable.txt"
old_mtime_ns = stable.stat().st_mtime_ns
os.utime(stable, ns=(old_mtime_ns + 1_000_000_000, old_mtime_ns + 1_000_000_000))

(api_root / "changed.txt").write_bytes(b"second version")
(api_root / "removed.txt").unlink()
(api_root / "new.txt").write_bytes(b"new bytes")

# reconcile
changes = catalog.reconcile_directory(api_root)
print("Summary:", changes.summary)
for change in changes.changes:
    previous = None if change.previous is None else change.previous.content_id.digest[:12]
    current = None if change.current is None else change.current.content_id.digest[:12]
    print(f"{change.relative_path.as_posix():12} {change.kind:9} reused={change.hash_reused} {previous} -> {current}")

print("Current catalog is still the old snapshot:", [item.relative_path.as_posix() for item in catalog.current_files(api_root)])


In [ ]:
api_files = catalog.current_files(api_root)
print(f"API current files: {len(api_files)}")
display(arc_pandas.current_files_frame(api_files))


In [ ]:
# accept changes to the catalog

catalog.apply_refresh(changes)
current = catalog.current_files(api_root)
print("Current paths after apply:", [item.relative_path.as_posix() for item in current])
print("Historical observations still include removed.txt:", any(
    observation.relative_path.as_posix() == "removed.txt" for observation in catalog.observation_history()
))


In [ ]:
api_files = catalog.current_files(api_root)
print(f"API current files: {len(api_files)}")
display(arc_pandas.current_files_frame(api_files))


## 4. The stale-baseline guard

A change set can be held for preview or later approval. If another successful refresh updates the catalog first, applying the older change set is rejected rather than overwriting current catalog state.


In [ ]:
stale_change_set = catalog.reconcile_directory(api_root)
(api_root / "later.txt").write_bytes(b"a later filesystem change")
catalog.scan_directory(api_root)  # This convenience method reconciles and applies immediately.

try:
    catalog.apply_refresh(stale_change_set)
except StaleRefreshError as error:
    print("Older change set rejected:", error)

print("Current paths remain:", [item.relative_path.as_posix() for item in catalog.current_files(api_root)])


## 5. Use the CLI

The CLI uses the same reconcile/apply workflow. It stores its catalog below the chosen root in `.archiver/catalog.sqlite`, which is automatically excluded from source observations.


In [ ]:
cli_root = workspace / "cli-source"
cli_root.mkdir()
(cli_root / "one.txt").write_bytes(b"one")

cli_command = [
    sys.executable,
    "-c",
    "from archiver.cli import main; raise SystemExit(main())",
]

def run_cli(*arguments: str) -> None:
    result = subprocess.run([*cli_command, "catalog", *arguments], text=True, capture_output=True, check=False)
    print("$ archiver catalog", " ".join(arguments))
    print(result.stdout, end="")
    if result.stderr:
        print(result.stderr, end="", file=sys.stderr)
    if result.returncode != 0:
        raise RuntimeError(f"CLI failed with exit code {result.returncode}")

run_cli("init", str(cli_root))
run_cli("scan", str(cli_root), "--no-progress")

(cli_root / "one.txt").write_bytes(b"one, updated")
(cli_root / "two.txt").write_bytes(b"two")
run_cli("scan", str(cli_root), "--no-progress")


## Exercise

Before running the answer, predict the change summary after deleting `two.txt` and reconciling. Remember: reconciliation is a dry run, so `catalog files` would still show the previous completed snapshot until an apply occurs.


In [ ]:
(cli_root / "two.txt").unlink()

with Catalog.open(cli_root / ".archiver" / "catalog.sqlite") as cli_catalog:
    preview = cli_catalog.reconcile_directory(cli_root, excluded_directories=(cli_root / ".archiver",))
    print("Predicted summary:", preview.summary)
    print("Catalog current paths are unchanged until apply:", [
        item.relative_path.as_posix() for item in cli_catalog.current_files(cli_root)
    ])


## Pitfalls and extensions

- **Metadata cache is not proof.** Equal path, size, and `mtime_ns` avoids hashing as a pragmatic optimization; deliberately restoring metadata can evade that fast path.
- **Changes during hashing.** For files that are hashed, Archiver compares regular-file metadata through the pathname and open descriptor before and after reading. A detectable mismatch aborts reconciliation, so it cannot be committed.
- **Filesystem changes after reconciliation.** Reconcile immediately before apply when freshness matters. The stale check only protects catalog state.

For a later feature, a UI or another caller can display `RefreshChangeSet` as a dry run, request approval, then call `apply_refresh()`—without adding a separate policy engine now.


In [ ]:
catalog.close()
shutil.rmtree(workspace)
print("Cleaned up:", workspace)
